In [ ]:
## knockdown code

In [ ]:
#aggreagation            env= dp_morning_repro

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
          "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
          "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

# Paths for the two separate outputs
OUTPUT_CSV_MEDIAN = os.path.join(PROJECT_ROOT,"files", "aggregated_wells_median.csv")
OUTPUT_CSV_STD = os.path.join(PROJECT_ROOT,"files", "aggregated_wells_std.csv")

CELL_COUNT_THRESHOLD = 0  
TREATMENT_COL = "Treatment"

all_plates_median = []
all_plates_std = []
all_cell_counts = [] 

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT, "features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT, "metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    # --- AGGREGATION & THRESHOLDING STEP ---
    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_cell_count = all_cells_in_well.shape[0]
        all_cell_counts.append(well_cell_count)

        if well_cell_count >= CELL_COUNT_THRESHOLD:
            # Calculate both Median and Std Dev
            well_median = np.median(all_cells_in_well, axis=0)
            well_std = np.std(all_cells_in_well, axis=0)
            
            base_info = {
                "Plate": plate_id, 
                "Well_ID": well_id, 
                "Treatment": well_to_treatment[well_id],
                "Cell_Count": well_cell_count
            }
            
            # Create rows for both dataframes
            row_median = base_info.copy()
            row_std = base_info.copy()
            
            for idx in range(len(well_median)):
                row_median[idx] = well_median[idx]
                row_std[idx] = well_std[idx]
                
            all_plates_median.append(row_median)
            all_plates_std.append(row_std)

# Convert to DataFrames
df_median = pd.DataFrame(all_plates_median)
df_std = pd.DataFrame(all_plates_std)

# Helper function to reorder
def reorder_cols(df):
    meta_cols = ["Plate", "Well_ID", "Treatment", "Cell_Count"]
    feat_cols = [c for c in df.columns if c not in meta_cols]
    return df[meta_cols + feat_cols]

df_median = reorder_cols(df_median)
df_std = reorder_cols(df_std)

print(f"\nAggregation complete.")

# ==========================================
# 2. VISUALIZATION: CELL COUNT HISTOGRAM
# ==========================================
counts = np.array(all_cell_counts)
c_mean = np.mean(counts)
c_median = np.median(counts)
c_std = np.std(counts)

plt.figure(figsize=(10, 6))
plt.hist(counts, bins=50, color='skyblue', edgecolor='black', alpha=0.7)

# Create stats text string
stats_text = f'Mean: {c_mean:.2f}\nMedian: {c_median:.2f}\nStd Dev: {c_std:.2f}'
# Place text box in the plot
plt.gca().text(0.95, 0.95, stats_text, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.title('Distribution of Cell Counts per Well (All Plates)')
plt.xlabel('Number of Cells')
plt.ylabel('Frequency (Wells)')
plt.grid(axis='y', alpha=0.3)
plt.show()

# ==========================================
# 3. SAVE TO CSVs
# ==========================================
df_median.to_csv(OUTPUT_CSV_MEDIAN, index=False)
df_std.to_csv(OUTPUT_CSV_STD, index=False)

print(f"Median data saved to: {OUTPUT_CSV_MEDIAN}")
print(f"Std Dev data saved to: {OUTPUT_CSV_STD}")

In [ ]:
#count patches

In [ ]:
import numpy as np
import os
from tqdm import tqdm

PROJECT_ROOT = r'E:\Thesis3april'
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
          "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
          "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

total_patches = 0
total_files = 0

print("Calculating total patch count...")

for plate in PLATES:
    feature_path = os.path.join(PROJECT_ROOT, "features", plate)
    
    if not os.path.exists(feature_path):
        print(f"Skipping {plate}: Path not found.")
        continue
        
    # Walk through all well subfolders
    for root, dirs, files in os.walk(feature_path):
        for file in files:
            if file.endswith(".npz"):
                file_path = os.path.join(root, file)
                try:
                    with np.load(file_path) as data:
                        # 'features' is the standard key in DeepProfiler npz files
                        # We only need the shape[0] (number of rows/cells)
                        total_patches += data["features"].shape[0]
                        total_files += 1
                except Exception as e:
                    print(f"Could not read {file}: {e}")

print("\n--- Final Statistics ---")
print(f"Total .npz files (sites) processed: {total_files}")
print(f"Total number of patches (cells):     {total_patches:,}")

In [ ]:
#remove mutants less than 5 patches

In [ ]:
import pandas as pd
import os

# 1. SETUP
PROJECT_ROOT = r'E:\Thesis3april'
INPUT_CSV = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "files")

# Create the directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. DEFINE YOUR NEW THRESHOLD
STRICT_THRESHOLD = 5 

# 3. LOAD & FILTER
print(f"Loading {INPUT_CSV}...")
df = pd.read_csv(INPUT_CSV)

# Identify the wells that ARE BELOW the threshold
removed_df = df[df['Cell_Count'] < STRICT_THRESHOLD].copy()

# Identify the wells that ARE ABOVE or EQUAL to the threshold
filtered_df = df[df['Cell_Count'] >= STRICT_THRESHOLD].copy()

# 4. REPORT & SAVE
print(f"\n--- Filtering Summary ---")
print(f"Original wells:       {len(df)}")
print(f"Wells kept:           {len(filtered_df)}")
print(f"Wells removed:        {len(removed_df)}")
print(f"-------------------------")

if not removed_df.empty:
    print(f"\n--- LIST OF REMOVED WELLS (Count < {STRICT_THRESHOLD}) ---")
    # We only show the metadata columns for the removed wells
    print(removed_df[['Plate', 'Well_ID', 'Treatment', 'Cell_Count']].to_string(index=False))
else:
    print("\nNo wells were below the threshold.")

# 5. SAVE DATA
OUTPUT_CSV_FILTERED = os.path.join(OUTPUT_DIR, f"aggregated_wells_median_min5.csv")
filtered_df.to_csv(OUTPUT_CSV_FILTERED, index=False)

print(f"\nDone! Filtered data saved to: {OUTPUT_CSV_FILTERED}")

In [ ]:
#plot without feature selection

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 0. ADAPTABLE STYLE PARAMETERS ---
# Axis Style
AXIS_LINE_WIDTH = 3       # Thickness of the L-shaped axis lines
AXIS_TICK_WIDTH = 3       # Thickness of the tick marks
AXIS_TICK_LEN = 8         # Length of the tick marks
AXIS_TICK_FONT_SIZE = 24  # Size of the numbers (0, 5, 10...) on the axis
AXIS_TITLE_FONT_SIZE = 24 # Size of "UMAP 1" and "UMAP 2" text

# Legend Style
LEGEND_FONT_SIZE = 24     # Font size for the legend text
LEGEND_SYMBOL_SIZE = 14   # Size of the symbols in the legend list
HEADER_SYMBOL_SIZE = 14   # Size of the Square/Circle symbols in the header

# Plot Marker Style
MARKER_SIZE = 8           # Size of the dots/squares in the actual plot
MARKER_OPACITY = 0.8      # Transparency of the points (0 to 1)

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "All_Plates_NofeatureSelection")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

# Create directories
for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)


df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# --- 2. SELECTION & MAPPING ---
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

time_map = {"T0": "no_xyl", "T1": "xyl5", "T2": "xyl10"}
unique_plates = sorted(df['Plate'].unique())
plate_rename_map = {}
for p in unique_plates:
    new_name = p.replace("PLATE", "P")
    for t_old, t_new in time_map.items():
        if t_old in new_name:
            new_name = new_name.replace(t_old, t_new)
    plate_rename_map[p] = new_name

df['Plate_Display'] = df['Plate'].map(plate_rename_map)

custom_colors = [
    '#5DADE2', '#D988B9', '#52BE80', '#E74C3C', '#AF7AC5', 
    '#707B7C', '#B7950B', '#8E44AD', '#E59896', '#5D3FD3', 
    '#45B39D', '#F333FF', '#FF3385', '#C39BD3', '#7D6608'
]

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Processing: {config['name']} ({len(valid_indices)} features)...")
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Save coordinates
    meta_cols = ['Plate', 'Plate_Display', 'Well_ID', 'Treatment', 'Type', 'Cell_Count']
    available_meta = [c for c in meta_cols if c in df.columns]
    df_coords = df[available_meta].copy()
    df_coords['UMAP1'], df_coords['UMAP2'] = embedding[:, 0], embedding[:, 1]
    df_coords.to_csv(os.path.join(COORD_DIR, f"Coords_{config['name'].replace(' ', '_')}.csv"), index=False)

    # --- CREATE PLOT ---
    fig = go.Figure()

    # Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no_sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # Data Traces
    conditions = sorted(df_coords['Plate_Display'].unique())
    for i, condition in enumerate(conditions):
        cond_data = df_coords[df_coords['Plate_Display'] == condition]
        color = custom_colors[i % len(custom_colors)]
        
        for t_type in ['Mutant', 'Control']:
            sub_data = cond_data[cond_data['Type'] == t_type]
            if sub_data.empty: continue
            
            fig.add_trace(go.Scatter(
                x=sub_data['UMAP1'], y=sub_data['UMAP2'],
                mode='markers', name=condition,
                showlegend=(t_type == 'Mutant'), 
                legendgroup=condition,
                marker=dict(
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY,
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0)
                ),
                text=sub_data['Treatment'], hoverinfo='text+name'
            ))

    # Layout
    fig.update_layout(
        width=900, height=900, template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE)
        ),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        )
    )

    # Save
    base_name = f"UMAP_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)
    
    print(f"Finished {config['name']}")

In [ ]:
#time overlay on everything without feature selection

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 0. ADAPTABLE STYLE PARAMETERS ---
# Axis Style
AXIS_LINE_WIDTH = 3       # Thickness of the L-shaped axis lines
AXIS_TICK_WIDTH = 3       # Thickness of the tick marks
AXIS_TICK_LEN = 8         # Length of the tick marks
AXIS_TICK_FONT_SIZE = 24  # Size of the numbers (0, 5, 10...) on the axis
AXIS_TITLE_FONT_SIZE = 24 # Size of "UMAP 1" and "UMAP 2" text

# Legend Style
LEGEND_FONT_SIZE = 24     # Font size for the legend text
LEGEND_SYMBOL_SIZE = 14   # Size of the symbols in the legend list
HEADER_SYMBOL_SIZE = 14   # Size of the Square/Circle symbols in the header

# Plot Marker Style
MARKER_SIZE = 8           # Size of the dots/squares in the actual plot
MARKER_OPACITY = 0.8      # Transparency of the points (0 to 1)

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "AllPlates_noSelection_Time_Overlay")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for folder in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# --- 2. PREPROCESSING & MAPPING ---
# Extract Timepoint from Plate name (T0, T1, T2)
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')

# Map to display names
time_display_map = {'T0': 'no_xyl', 'T1': 'xyl5', 'T2': 'xyl10'}
df['Time_Display'] = df['Timepoint'].map(time_display_map)

# Define Treatment types
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# Color Mapping for Timepoints
time_colors = {
    'no_xyl': '#FF7F50', # Coral
    'xyl5': '#008080',   # Teal
    'xyl10': '#DAA520'   # Goldenrod
}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Generating Time Overlay for: {config['name']}...")
    
    # Scale and Reduce
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Prep plotting data
    df_plot = df[['Plate', 'Well_ID', 'Treatment', 'Type', 'Time_Display']].copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # Save Coordinates
    coord_file = f"Coords_Time_{config['name'].replace(' ', '_')}.csv"
    df_plot.to_csv(os.path.join(COORD_DIR, coord_file), index=False)

    # Initialize Figure
    fig = go.Figure()

    # A) Add Legend Header (Dummy Traces)
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no_sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # B) Add Data Traces grouped by Time
    times = ['no_xyl', 'xyl5', 'xyl10']
    for t_val in times:
        for t_type in ['Mutant', 'Control']:
            mask = (df_plot['Time_Display'] == t_val) & (df_plot['Type'] == t_type)
            curr = df_plot[mask]
            if curr.empty: continue
            
            color = time_colors[t_val]
            
            fig.add_trace(go.Scatter(
                x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                name=t_val,
                legendgroup=t_val,
                # Only show the name in the legend for the first marker type (Mutant)
                showlegend=True if t_type == 'Mutant' else False,
                marker=dict(
                    color=color, 
                    size=MARKER_SIZE,
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0),
                    opacity=MARKER_OPACITY
                ),
                text=curr['Treatment'],
                hoverinfo='text+name'
            ))

    # C) Final Layout
    fig.update_layout(
        width=900, height=900,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE)
        ),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        )
    )

    # Save
    base_name = f"UMAP_Time_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)

print("\nDone. Time overlay plots and coordinates are saved.")

In [ ]:
#feature seleciton on everything 

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
INPUT_CSV = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "files")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
df_raw.columns = [str(c) for c in df_raw.columns]

# Separate Metadata and Features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    
    # Selection based on batch noise ranking
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# --- STEP 0: GLOBAL VARIANCE FILTER ---
# Removes features that are flat/near-zero across the whole experiment first
print(f"\nRunning Step 0: Global Variance Filter (std > 0.01)...")
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
print(f"Removed {len(feature_cols) - len(active_features)} low-variance features.")

# --- STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"Running Step 1: Within-Plate Consistency (Filtering to top 2000)...")
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=2000)
df_step1 = df_raw[metadata_cols + step1_features]

# --- STEP 2: ACROSS-PLATE STABILITY ---
print(f"Running Step 2: Across-Plate Stability (Filtering to top 200)...")
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
df_step2 = df_step1[metadata_cols + step2_features]

# --- STEP 3: REDUNDANCY REMOVAL ---
print(f"Running Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final = df_step2[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Original features: {len(feature_cols)}")
print(f"Active features (Step 0): {len(active_features)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#Plotting all after featue seleciton

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 0. ADAPTABLE STYLE PARAMETERS ---
# Axis Style
AXIS_LINE_WIDTH = 3       # Thickness of the L-shaped axis lines
AXIS_TICK_WIDTH = 3       # Thickness of the tick marks
AXIS_TICK_LEN = 8         # Length of the tick marks
AXIS_TICK_FONT_SIZE = 24  # Size of the numbers (0, 5, 10...) on the axis
AXIS_TITLE_FONT_SIZE = 24 # Size of "UMAP 1" and "UMAP 2" text

# Legend Style
LEGEND_FONT_SIZE = 24     # Font size for the legend text
LEGEND_SYMBOL_SIZE = 14   # Size of the symbols in the legend list
HEADER_SYMBOL_SIZE = 14   # Size of the Square/Circle symbols in the header

# Plot Marker Style
MARKER_SIZE = 8           # Size of the dots/squares in the actual plot
MARKER_OPACITY = 0.8      # Transparency of the points (0 to 1)

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "All_Plates_featureSelection")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# --- 2. SELECTION & MAPPING ---
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

time_map = {"T0": "no_xyl", "T1": "xyl5", "T2": "xyl10"}
unique_plates = sorted(df['Plate'].unique())
plate_rename_map = {}
for p in unique_plates:
    new_name = p.replace("PLATE", "P")
    for t_old, t_new in time_map.items():
        if t_old in new_name:
            new_name = new_name.replace(t_old, t_new)
    plate_rename_map[p] = new_name

df['Plate_Display'] = df['Plate'].map(plate_rename_map)

custom_colors = [
    '#5DADE2', '#D988B9', '#52BE80', '#E74C3C', '#AF7AC5', 
    '#707B7C', '#B7950B', '#8E44AD', '#E59896', '#5D3FD3', 
    '#45B39D', '#F333FF', '#FF3385', '#C39BD3', '#7D6608'
]

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Processing: {config['name']} ({len(valid_indices)} features)...")
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Save coordinates
    meta_cols = ['Plate', 'Plate_Display', 'Well_ID', 'Treatment', 'Type', 'Cell_Count']
    available_meta = [c for c in meta_cols if c in df.columns]
    df_coords = df[available_meta].copy()
    df_coords['UMAP1'], df_coords['UMAP2'] = embedding[:, 0], embedding[:, 1]
    df_coords.to_csv(os.path.join(COORD_DIR, f"Coords_{config['name'].replace(' ', '_')}.csv"), index=False)

    # --- CREATE PLOT ---
    fig = go.Figure()

    # Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no_sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # Data Traces
    conditions = sorted(df_coords['Plate_Display'].unique())
    for i, condition in enumerate(conditions):
        cond_data = df_coords[df_coords['Plate_Display'] == condition]
        color = custom_colors[i % len(custom_colors)]
        
        for t_type in ['Mutant', 'Control']:
            sub_data = cond_data[cond_data['Type'] == t_type]
            if sub_data.empty: continue
            
            fig.add_trace(go.Scatter(
                x=sub_data['UMAP1'], y=sub_data['UMAP2'],
                mode='markers', name=condition,
                showlegend=(t_type == 'Mutant'), 
                legendgroup=condition,
                marker=dict(
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY,
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0)
                ),
                text=sub_data['Treatment'], hoverinfo='text+name'
            ))

    # Layout
    fig.update_layout(
        width=900, height=900, template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE)
        ),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        )
    )

    # Save
    base_name = f"UMAP_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=2)
    
    print(f"Finished {config['name']}")

In [ ]:
#all met iets lichtere gedient maar mschn minder goed

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.colors as mc
import colorsys

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 24     
LEGEND_SYMBOL_SIZE = 14   
HEADER_SYMBOL_SIZE = 14   

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      

# Helper function to lighten/darken colors
def adjust_color(color, amount=0.5):
    """
    Lightens the given color by multiplying (1-luminance) by the amount.
    Input can be matplotlib color string, hex string, or RGB tuple.
    amount > 1.0 = lighter; amount < 1.0 = darker.
    """
    try:
        c = mc.cnames[color]
    except:
        c = color
    c = colorsys.rgb_to_hls(*mc.to_rgb(c))
    return mc.to_hex(colorsys.hls_to_rgb(c[0], max(0, min(1, amount * c[1])), c[2]))

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "All_Plates_featureSelection")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# --- 2. SELECTION & MAPPING ---
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

time_map = {"T0": "no_xyl", "T1": "xyl5", "T2": "xyl10"}
unique_plates = sorted(df['Plate'].unique())
plate_rename_map = {}
for p in unique_plates:
    new_name = p.replace("PLATE", "P")
    for t_old, t_new in time_map.items():
        if t_old in new_name:
            new_name = new_name.replace(t_old, t_new)
    plate_rename_map[p] = new_name

df['Plate_Display'] = df['Plate'].map(plate_rename_map)

# Your base colors
custom_colors = [
    '#5DADE2', '#D988B9', '#52BE80', '#E74C3C', '#AF7AC5', 
    '#707B7C', '#B7950B', '#8E44AD', '#E59896', '#5D3FD3', 
    '#45B39D', '#F333FF', '#FF3385', '#C39BD3', '#7D6608'
]

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Processing: {config['name']} ({len(valid_indices)} features)...")
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_coords = df[['Plate', 'Plate_Display', 'Well_ID', 'Treatment', 'Type', 'Cell_Count']].copy()
    df_coords['UMAP1'], df_coords['UMAP2'] = embedding[:, 0], embedding[:, 1]
    df_coords.to_csv(os.path.join(COORD_DIR, f"Coords_{config['name'].replace(' ', '_')}.csv"), index=False)

    fig = go.Figure()

    # Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no_sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # Data Traces
    conditions = sorted(df_coords['Plate_Display'].unique())
    for i, condition in enumerate(conditions):
        cond_data = df_coords[df_coords['Plate_Display'] == condition]
        base_color = custom_colors[i % len(custom_colors)]
        
        # Mutant = Lighter, Control = Darker
        mutant_color = adjust_color(base_color, amount=1.2) 
        control_color = adjust_color(base_color, amount=0.7) 
        
        for t_type in ['Mutant', 'Control']:
            sub_data = cond_data[cond_data['Type'] == t_type]
            if sub_data.empty: continue
            
            color = mutant_color if t_type == 'Mutant' else control_color
            
            fig.add_trace(go.Scatter(
                x=sub_data['UMAP1'], y=sub_data['UMAP2'],
                mode='markers', name=condition,
                showlegend=(t_type == 'Mutant'), 
                legendgroup=condition,
                marker=dict(
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY,
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0)
                ),
                text=sub_data['Treatment'], hoverinfo='text+name'
            ))

    fig.update_layout(
        width=900, height=900, template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(itemsizing='constant', font=dict(size=LEGEND_FONT_SIZE)),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        )
    )

    base_name = f"UMAP_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    # Updated to scale=7 for 500 DPI requirement
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)
    
    print(f"Finished {config['name']}")

In [ ]:
#time overlay

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 24     
LEGEND_SYMBOL_SIZE = 14   
HEADER_SYMBOL_SIZE = 14   

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "All_Plates_FeatureSelection_Time_Overlay")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# --- 2. PREPROCESSING & MAPPING ---
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
time_display_map = {'T0': 'no_xyl', 'T1': 'xyl5', 'T2': 'xyl10'}
df['Time_Display'] = df['Timepoint'].map(time_display_map)

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# Color Mapping: Sub-divided into Mutant (Lighter) and Control (Darker)
# Base Coral: #FF7F50 | Base Teal: #008080 | Base Goldenrod: #DAA520
time_colors = {
    'no_xyl': {'Mutant': '#FFA07A', 'Control': '#E9967A'}, # Light Salmon vs Darker Coral
    'xyl5':   {'Mutant': '#4DB6AC', 'Control': '#00695C'}, # Light Teal vs Deep Teal
    'xyl10':  {'Mutant': '#F0D05D', 'Control': '#B8860B'}  # Light Goldenrod vs Dark Goldenrod
}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Generating Time Overlay for: {config['name']}...")
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_plot = df[['Plate', 'Well_ID', 'Treatment', 'Type', 'Time_Display']].copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    coord_file = f"Coords_Time_{config['name'].replace(' ', '_')}.csv"
    df_plot.to_csv(os.path.join(COORD_DIR, coord_file), index=False)

    fig = go.Figure()

    # A) Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no_sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # B) Data Traces
    times = ['no_xyl', 'xyl5', 'xyl10']
    for t_val in times:
        for t_type in ['Mutant', 'Control']:
            mask = (df_plot['Time_Display'] == t_val) & (df_plot['Type'] == t_type)
            curr = df_plot[mask]
            if curr.empty: continue
            
            # Select the specific shade for this type
            color = time_colors[t_val][t_type]
            
            fig.add_trace(go.Scatter(
                x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                name=t_val,
                legendgroup=t_val,
                showlegend=True if t_type == 'Mutant' else False,
                marker=dict(
                    color=color, 
                    size=MARKER_SIZE,
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    # Controls have a black outline to make them pop even more
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0),
                    opacity=MARKER_OPACITY
                ),
                text=curr['Treatment'],
                hovertemplate="<b>%{text}</b><br>Condition: %{name}<extra></extra>"
            ))

    # C) Final Layout
    fig.update_layout(
        width=900, height=900,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE)
        ),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        )
    )

    base_name = f"UMAP_Time_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    # scale=7 ensures ~500 DPI for a 900x900 base image
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)

print("\nDone. 500 DPI PNGs, SVGs, and coordinates are saved.")

In [ ]:

# 3.5 FEATURE SUMMARY PRINT-OUT
print("\n" + "="*45)
print(f"{'Channel Selection':<25} | {'Features Found':<15}")
print("-" * 45)
for config in plot_configs:
    print(f"{config['name']:<25} | {len(config['indices']):<15}")
print("="*45 + "\n")

In [ ]:
#enkelT0enT1 bekijken
###
#

In [ ]:
#preselection T0 en T1 enkel 

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
INPUT_CSV = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "files")
CONTROL_LABEL = "no_sgRNA" 

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Loading raw data...")
df_raw_full = pd.read_csv(INPUT_CSV)
df_raw_full.columns = [str(c) for c in df_raw_full.columns]

# --- NEW STEP: FILTER OUT T2 AND SAVE COPY ---
print("Filtering out T2 samples...")
# This assumes your Plate column ends with the timepoint (e.g., 'PlateName_T0')
df_raw = df_raw_full[df_raw_full['Plate'].str.contains('T0|T1')].copy()

# Save the T0_T1 raw subset
filtered_raw_path = os.path.join(OUTPUT_DIR, "raw_data_T0_T1_only.csv")
df_raw.to_csv(filtered_raw_path, index=False)
print(f"Saved filtered raw copy to: {filtered_raw_path}")

# Separate Metadata and Features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    # UPDATED: Removed 'T2' from the loop
    for tp in ['T0', 'T1']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        print("Warning: Not enough plates per timepoint to calculate stability. Returning input features.")
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# --- STEP 0: GLOBAL VARIANCE FILTER ---
print(f"\nRunning Step 0: Global Variance Filter (std > 0.01)...")
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
print(f"Removed {len(feature_cols) - len(active_features)} low-variance features.")

# --- STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"Running Step 1: Within-Plate Consistency (Filtering to top 2000)...")
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=2000)
df_step1 = df_raw[metadata_cols + step1_features]

# --- STEP 2: ACROSS-PLATE STABILITY ---
print(f"Running Step 2: Across-Plate Stability (Filtering to top 200)...")
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
df_step2 = df_step1[metadata_cols + step2_features]

# --- STEP 3: REDUNDANCY REMOVAL ---
print(f"Running Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final = df_step2[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts_T0_T1.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE (T0 & T1 ONLY)")
print(f"Original rows (including T2): {len(df_raw_full)}")
print(f"Filtered rows (T0 & T1): {len(df_raw)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved results to: {output_path}")
print("="*40)

In [ ]:
#plot enkel T0 en T1

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.colors as mc
import colorsys

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 24     
LEGEND_SYMBOL_SIZE = 14   
HEADER_SYMBOL_SIZE = 14   

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      

# Helper function to lighten/darken colors
def adjust_color(color, amount=0.5):
    try:
        c = mc.cnames[color]
    except:
        c = color
    c = colorsys.rgb_to_hls(*mc.to_rgb(c))
    return mc.to_hex(colorsys.hls_to_rgb(c[0], max(0, min(1, amount * c[1])), c[2]))

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts_T0_T1.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "T0T1_featureSelection")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# Filter for T0 and T1 only
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1","PLATE3_T0","PLATE3_T1",
                   "PLATE4_T0","PLATE4_T1","PLATE5_T0","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

time_map = {"T0": "no_xyl", "T1": "xyl5", "T2": "xyl10"}
unique_plates = sorted(df['Plate'].unique())
plate_rename_map = {}
for p in unique_plates:
    new_name = p.replace("PLATE", "P")
    for t_old, t_new in time_map.items():
        if t_old in new_name:
            new_name = new_name.replace(t_old, t_new)
    plate_rename_map[p] = new_name

df['Plate_Display'] = df['Plate'].map(plate_rename_map)

# NEW: Optimized 10-color palette for 5 plates x 2 timepoints
custom_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
]

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Processing: {config['name']} ({len(valid_indices)} features)...")
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_coords = df[['Plate', 'Plate_Display', 'Well_ID', 'Treatment', 'Type', 'Cell_Count']].copy()
    df_coords['UMAP1'], df_coords['UMAP2'] = embedding[:, 0], embedding[:, 1]
    df_coords.to_csv(os.path.join(COORD_DIR, f"Coords_{config['name'].replace(' ', '_')}.csv"), index=False)

    fig = go.Figure()

    # Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no_sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # Data Traces
    conditions = sorted(df_coords['Plate_Display'].unique())
    for i, condition in enumerate(conditions):
        cond_data = df_coords[df_coords['Plate_Display'] == condition]
        base_color = custom_colors[i % len(custom_colors)]
        
        # Mutant = Lighter, Control = Darker
        mutant_color = adjust_color(base_color, amount=1.2) 
        control_color = adjust_color(base_color, amount=0.7) 
        
        for t_type in ['Mutant', 'Control']:
            sub_data = cond_data[cond_data['Type'] == t_type]
            if sub_data.empty: continue
            
            color = mutant_color if t_type == 'Mutant' else control_color
            
            fig.add_trace(go.Scatter(
                x=sub_data['UMAP1'], y=sub_data['UMAP2'],
                mode='markers', name=condition,
                showlegend=(t_type == 'Mutant'), 
                legendgroup=condition,
                marker=dict(
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY,
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0)
                ),
                text=sub_data['Treatment'], hoverinfo='text+name'
            ))

    fig.update_layout(
        width=900, height=900, template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(itemsizing='constant', font=dict(size=LEGEND_FONT_SIZE)),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        )
    )

    base_name = f"UMAP_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)
    
    print(f"Finished {config['name']}")

print("\nDone. T0 and T1 plots generated with 500 DPI quality.")

In [ ]:

# 3.5 FEATURE SUMMARY PRINT-OUT
print("\n" + "="*45)
print(f"{'Channel Selection':<25} | {'Features Found':<15}")
print("-" * 45)
for config in plot_configs:
    print(f"{config['name']:<25} | {len(config['indices']):<15}")
print("="*45 + "\n")

In [ ]:
#time overlay T0 en T1

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 24     
LEGEND_SYMBOL_SIZE = 14   
HEADER_SYMBOL_SIZE = 14   

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts_T0_T1.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "TOT1_FeatureSelection_Time_Overlay")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# --- 2. PREPROCESSING & MAPPING ---
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
time_display_map = {'T0': 'no_xyl', 'T1': 'xyl5', 'T2': 'xyl10'}
df['Time_Display'] = df['Timepoint'].map(time_display_map)

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# Color Mapping: Sub-divided into Mutant (Lighter) and Control (Darker)
# Base Coral: #FF7F50 | Base Teal: #008080 | Base Goldenrod: #DAA520
time_colors = {
    'no_xyl': {'Mutant': '#FFA07A', 'Control': '#E9967A'}, # Light Salmon vs Darker Coral
    'xyl5':   {'Mutant': '#4DB6AC', 'Control': '#00695C'}, # Light Teal vs Deep Teal
      # Light Goldenrod vs Dark Goldenrod
}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Generating Time Overlay for: {config['name']}...")
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_plot = df[['Plate', 'Well_ID', 'Treatment', 'Type', 'Time_Display']].copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    coord_file = f"Coords_Time_{config['name'].replace(' ', '_')}.csv"
    df_plot.to_csv(os.path.join(COORD_DIR, coord_file), index=False)

    fig = go.Figure()

    # A) Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no_sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # B) Data Traces
    times = ['no_xyl', 'xyl5', 'xyl10']
    for t_val in times:
        for t_type in ['Mutant', 'Control']:
            mask = (df_plot['Time_Display'] == t_val) & (df_plot['Type'] == t_type)
            curr = df_plot[mask]
            if curr.empty: continue
            
            # Select the specific shade for this type
            color = time_colors[t_val][t_type]
            
            fig.add_trace(go.Scatter(
                x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                name=t_val,
                legendgroup=t_val,
                showlegend=True if t_type == 'Mutant' else False,
                marker=dict(
                    color=color, 
                    size=MARKER_SIZE,
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    # Controls have a black outline to make them pop even more
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0),
                    opacity=MARKER_OPACITY
                ),
                text=curr['Treatment'],
                hovertemplate="<b>%{text}</b><br>Condition: %{name}<extra></extra>"
            ))

    # C) Final Layout
    fig.update_layout(
        width=900, height=900,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE)
        ),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        )
    )

    base_name = f"UMAP_Time_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    # scale=7 ensures ~500 DPI for a 900x900 base image
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)

print("\nDone. 500 DPI PNGs, SVGs, and coordinates are saved.")

In [ ]:
#annotation T0T1, met nieuwe kleur

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 0. ADAPTABLE STYLE PARAMETERS ---
# Axis Style
AXIS_LINE_WIDTH = 3       # Thickness of the L-shaped axis lines
AXIS_TICK_WIDTH = 3       # Thickness of the tick marks
AXIS_TICK_LEN = 8         # Length of the tick marks
AXIS_TICK_FONT_SIZE = 24  # Size of the numbers (0, 5, 10...) on the axis
AXIS_TITLE_FONT_SIZE = 24 # Size of "UMAP 1" and "UMAP 2" text

# Legend Style
LEGEND_FONT_SIZE = 14     # Font size for the legend text
LEGEND_TITLE_SIZE = 16    # Font size for the 'Pathways' header
LEGEND_SYMBOL_SIZE = 14   # Size of the symbols in the legend list

# Plot Marker Style
MARKER_SIZE_MUTANT = 8    # Size of the dots
MARKER_SIZE_CONTROL = 10  # Size of the squares/diamonds
MARKER_OPACITY = 0.8      # Transparency of the points (0 to 1)

# Export Quality
PNG_RESOLUTION_SCALE = 7  # scale=7 ensures ~500 DPI for a 900x900 base image

# --- 1. SETUP & THE COLOR DICTIONARY ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts_T0_T1.csv")
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "T0T1_Annotated_Channels")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

# EXPLICIT COLOR MAP
MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                      
    "biosynthesis of fatty acids": "#9467bd",    
    "DNA replication": "#8c564b",                
    "DNA condensation/ segregation": "#e377c2",  
    "biosynthesis of isoprenoids": "#7f7f7f",    
    "cell division": "#bcbd22",                  
    "ribosomal proteins": "#17becf",             
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                      
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#333333",  
    "Baseline (T0)": "#808080",  
    "Unknown/Other": "#D3D3D3"   
}

# --- 2. DATA LOADING & ANNOTATION ---
df_raw = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df_raw.columns = [str(c) for c in df_raw.columns]

df = df_raw.copy()
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

df['SubtiWiki Annotation 4'] = df['SubtiWiki Annotation 4'].astype(str).str.strip()
df['SubtiWiki Annotation 3'] = df['SubtiWiki Annotation 3'].astype(str).str.strip()

df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].replace('nan', np.nan).fillna(df['SubtiWiki Annotation 3'].replace('nan', np.nan)).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

is_ctrl = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[(df['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
df.loc[is_ctrl, 'Display_Category'] = 'Control Group'

# Final Legend Order logic
unique_in_data = df['Display_Category'].unique()
other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["Unknown/Other"]

auto_pal = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
color_lookup = MANUAL_COLORS.copy()
for i, cat in enumerate(other_cats):
    color_lookup[cat] = auto_pal[i % len(auto_pal)]

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df_raw.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Processing Channel: {config['name']} ({len(valid_indices)} features)...")
    
    # Scale and Reduce
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    # Create copy for plotting and add coords
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # Save Coordinates (Metadata only)
    meta_cols = ['Plate', 'Well_ID', 'Treatment', 'Timepoint', 'Display_Category', 'Cell_Count', 'UMAP1', 'UMAP2']
    available_meta = [c for c in meta_cols if c in df_plot.columns]
    df_plot[available_meta].to_csv(os.path.join(COORD_DIR, f"Coords_{config['name'].replace(' ', '_')}.csv"), index=False)

    # Plotting
    fig = go.Figure()

    # Draw in reverse for layering
    for cat in final_legend_order[::-1]:
        if cat not in df_plot['Display_Category'].values: continue
        cat_df = df_plot[df_plot['Display_Category'] == cat]
        is_c_cat = (cat == "Control Group")

        # Split by symbol groups
        sub_groups = [
            ('circle', cat_df[~cat_df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])]),
            ('square', cat_df[(cat_df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])) & (cat_df['Timepoint'] == 'T0')]),
            ('diamond', cat_df[(cat_df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])) & (cat_df['Timepoint'] == 'T1')])
        ]

        for sym, sub_df in sub_groups:
            if sub_df.empty: continue
            sz = MARKER_SIZE_CONTROL if is_c_cat else MARKER_SIZE_MUTANT
            clr = color_lookup.get(cat, "#000000")

            # Generate Rich Metadata Hover
            hover_text = []
            for _, r in sub_df.iterrows():
                label = (
                    f"<b>Category:</b> {cat}<br>"
                    f"<b>Treatment:</b> {r.get('Treatment', 'N/A')}<br>"
                    f"<b>Timepoint:</b> {r.get('Timepoint', 'N/A')}<br>"
                    f"<b>Plate:</b> {r.get('Plate', 'N/A')}<br>"
                    f"<b>Well:</b> {r.get('Well_ID', 'N/A')}<br>"
                    f"<b>Cell Count:</b> {r.get('Cell_Count', 'N/A')}"
                )
                hover_text.append(label)

            fig.add_trace(go.Scatter(
                x=sub_df['UMAP1'], y=sub_df['UMAP2'],
                mode='markers', name=cat, legendgroup=cat, showlegend=False,
                marker=dict(
                    size=sz, color=clr, symbol=sym,
                    opacity=MARKER_OPACITY + 0.1 if is_c_cat else MARKER_OPACITY,
                    line=dict(width=0.4, color='white')
                ),
                text=hover_text, hoverinfo='text'
            ))

    # Legend override
    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            if trace.name == cat and cat not in seen:
                trace.showlegend = True
                seen.add(cat)
                trace.legendrank = final_legend_order.index(cat)

    # Layout
    fig.update_layout(
        template='plotly_white', width=1350, height=900,
        margin=dict(l=80, r=20, b=80, t=80), font=dict(family="Arial"),
        legend=dict(title_text='<b>Pathways</b>', title_font=dict(size=LEGEND_TITLE_SIZE),
                    x=1.02, y=1, font=dict(size=LEGEND_FONT_SIZE), itemsizing='constant'),
        xaxis=dict(title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                   showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, showgrid=False,
                   ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
                   tickfont=dict(size=AXIS_TICK_FONT_SIZE), scaleanchor="y", scaleratio=1),
        yaxis=dict(title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                   showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, showgrid=False,
                   ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
                   tickfont=dict(size=AXIS_TICK_FONT_SIZE))
    )

    # Save
    base_name = f"UMAP_Annotated_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=PNG_RESOLUTION_SCALE)

print("\nDone! All channel plots (HTML, SVG, PNG) and coordinates have been saved.")

In [ ]:
#mooiste contour

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt
from matplotlib.path import Path

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 14     
LEGEND_TITLE_SIZE = 16    

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      
PNG_RESOLUTION_SCALE = 7  # 500 DPI approx

# --- 1. SETUP & THE COLOR DICTIONARY ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts_T0_T1.csv")
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "T0T1_Contour")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                         
    "biosynthesis of fatty acids": "#9467bd",     
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "biosynthesis of isoprenoids": "#7f7f7f",     
    "cell division": "#bcbd22",                   
    "ribosomal proteins": "#17becf",              
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                       
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#808080",  
    "Baseline (T0)": "#808080",  
    "Unknown/Other": "#D3D3D3"   
}

# --- 2. DATA LOADING & ANNOTATION ---
df_raw = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df_raw.columns = [str(c) for c in df_raw.columns]

df = df_raw.copy()
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

df['SubtiWiki Annotation 4'] = df['SubtiWiki Annotation 4'].astype(str).str.strip()
df['SubtiWiki Annotation 3'] = df['SubtiWiki Annotation 3'].astype(str).str.strip()

df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].replace('nan', np.nan).fillna(df['SubtiWiki Annotation 3'].replace('nan', np.nan)).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

is_ctrl = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[(df['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
df.loc[is_ctrl, 'Display_Category'] = 'Control Group'

# Legend Order Logic
unique_in_data = df['Display_Category'].unique()
other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["Unknown/Other"]

auto_pal = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
color_lookup = MANUAL_COLORS.copy()
for i, cat in enumerate(other_cats):
    color_lookup[cat] = auto_pal[i % len(auto_pal)]

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df_raw.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Processing Logic Percentages for: {config['name']}...")
    
    # UMAP
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Save Coords
    meta_cols = ['Plate', 'Well_ID', 'Treatment', 'Timepoint', 'Display_Category', 'Cell_Count', 'UMAP1', 'UMAP2']
    df_plot[[c for c in meta_cols if c in df_plot.columns]].to_csv(os.path.join(COORD_DIR, f"Coords_Logic_{config['name'].replace(' ', '_')}.csv"), index=False)

    fig = go.Figure()

    # --- 5. KDE Logic Percentages ---
    ctrl_df = df_plot[df_plot['Display_Category'] == 'Control Group']
    if not ctrl_df.empty:
        x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
        points_array = np.vstack([x_pts, y_pts]).T
        kde = st.gaussian_kde(points_array.T)
        point_densities = kde(points_array.T)
        
        target_inclusions = [95, 70, 45, 20]
        percentile_thresholds = [100 - t for t in target_inclusions]
        levels = [np.percentile(point_densities, p) for p in percentile_thresholds]
        
        pad = 2
        x_grid = np.linspace(x_pts.min() - pad, x_pts.max() + pad, 150)
        y_grid = np.linspace(y_pts.min() - pad, y_pts.max() + pad, 150)
        X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
        Z_mesh = kde(np.vstack([X_mesh.ravel(), Y_mesh.ravel()])).reshape(X_mesh.shape)

        fig_temp, ax_temp = plt.subplots()
        cs = ax_temp.contour(X_mesh, Y_mesh, Z_mesh, levels=levels)
        
        for i, (level_val, target_pct) in enumerate(zip(levels, target_inclusions)):
            segments = cs.allsegs[i]
            for j, seg in enumerate(segments):
                fig.add_trace(go.Scatter(
                    x=seg[:, 0], y=seg[:, 1], mode='lines',
                    line=dict(color='#808080', width=1.2),
                    name=f'Control Group ({target_pct}%)' if i == 0 and j == 0 else 'Control_Inner',
                    legendgroup='Control Group', showlegend=False, hoverinfo='skip'
                ))
        plt.close(fig_temp)

    # --- 6. Plot Categories with Rich Hover ---
    remaining_cats = [c for c in final_legend_order[::-1] if c != "Control Group"]

    for cat in remaining_cats:
        if cat not in df_plot['Display_Category'].values: continue
        sub_df = df_plot[df_plot['Display_Category'] == cat]
        clr = color_lookup.get(cat, "#000000")

        # Generate Hover Label
        hover_labels = []
        for _, r in sub_df.iterrows():
            txt = (f"<b>Category:</b> {cat}<br>"
                   f"<b>Treatment:</b> {r.get('Treatment','N/A')}<br>"
                   f"<b>Timepoint:</b> {r.get('Timepoint','N/A')}<br>"
                   f"<b>Plate:</b> {r.get('Plate','N/A')}<br>"
                   f"<b>Well:</b> {r.get('Well_ID','N/A')}<br>"
                   f"<b>Cell Count:</b> {r.get('Cell_Count','N/A')}")
            hover_labels.append(txt)

        fig.add_trace(go.Scatter(
            x=sub_df['UMAP1'], y=sub_df['UMAP2'], mode='markers',
            name=cat, legendgroup=cat, showlegend=False,
            marker=dict(size=MARKER_SIZE, color=clr, symbol='circle',
                        opacity=MARKER_OPACITY, line=dict(width=0.4, color='white')),
            text=hover_labels, hoverinfo='text'
        ))

    # --- 7. Legend Order Override ---
    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            name = trace.name if trace.name else ""
            if name.startswith('Control Group') and 'Control Group' not in seen:
                trace.showlegend = True
                trace.name = 'Control Group'
                seen.add('Control Group')
                trace.legendrank = final_legend_order.index('Control Group')
            elif name == cat and cat not in seen:
                trace.showlegend = True
                seen.add(cat)
                trace.legendrank = final_legend_order.index(cat)

    # --- 8. Layout ---
    fig.update_layout(
        template='plotly_white', width=1350, height=900,
        margin=dict(l=80, r=20, b=80, t=80), font=dict(family="Arial"),
        legend=dict(title_text='<b>Pathways</b>', title_font=dict(size=LEGEND_TITLE_SIZE),
                    x=1.02, y=1, font=dict(size=LEGEND_FONT_SIZE), itemsizing='constant'),
        xaxis=dict(title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                   showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, showgrid=False,
                   ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
                   tickfont=dict(size=AXIS_TICK_FONT_SIZE), scaleanchor="y", scaleratio=1),
        yaxis=dict(title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                   showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, showgrid=False,
                   ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
                   tickfont=dict(size=AXIS_TICK_FONT_SIZE))
    )

    base_name = f"UMAP_Logic_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=PNG_RESOLUTION_SCALE)

print("\nDone! All channels processed with KDE Logic Percentages.")

In [ ]:
#area overlay: 3 april still to be adapted

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# ==========================================
# 1. SETUP & DATA MAPPING
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'

file_path = os.path.join(PROJECT_ROOT, "2april", "vettedcellcounts_2april_T0_T1.csv")
df = pd.read_csv(file_path)

df.columns = [str(c) for c in df.columns]

# --- LOAD AUC DATA WITH YOUR PATHS ---
auc_no_xylose_path = r'C:\Users\arnou\Documents\thesis\Resultaten\GrowthCurves\AUC_without_xylose.csv' 
auc_with_xylose_path = r'C:\Users\arnou\Documents\thesis\Resultaten\GrowthCurves\AUC_with_xylose.csv'

auc_no_xylose = pd.read_csv(auc_no_xylose_path)
auc_with_xylose = pd.read_csv(auc_with_xylose_path)

map_no_xylose = dict(zip(auc_no_xylose['Gene_target'], auc_no_xylose['AUC']))
map_with_xylose = dict(zip(auc_with_xylose['Gene_target'], auc_with_xylose['AUC']))

def assign_auc(row):
    treatment = str(row['Treatment'])
    plate = str(row['Plate'])
    # Return NaN if missing to keep it out of the color scale
    if "_T0" in plate:
        return map_no_xylose.get(treatment, np.nan)
    else:
        return map_with_xylose.get(treatment, np.nan)

# Create AUC column (NaNs preserved for grey coloring)
df['AUC'] = df.apply(assign_auc, axis=1)

# Identify 'Type' for symbols
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2april", "UMAP_2april_AUC")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ==========================================
# 2. FEATURE SELECTION
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 3. RUN UMAP & PLOT
# ==========================================
for config in plot_configs:
    if not config['indices']:
        continue
    
    print(f"Processing UMAP for: {config['name']}...")
    
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Split data to protect the color scale range
    df_valid = df_plot[df_plot['AUC'].notna()]
    df_missing = df_plot[df_plot['AUC'].isna()]
    
    # Trace 1: Valid AUC (Viridis Scale)
    fig = px.scatter(
        df_valid, 
        x='UMAP1', 
        y='UMAP2', 
        color='AUC',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'AUC': ':.4f', 'UMAP1': False, 'UMAP2': False},
        color_continuous_scale='Viridis',
        title=f"UMAP: {config['name']} (Grey = Missing AUC)",
        template='plotly_white'
    )
    
    # Trace 2: Missing AUC (Static Grey)
    fig.add_trace(
        go.Scatter(
            x=df_missing['UMAP1'],
            y=df_missing['UMAP2'],
            mode='markers',
            marker=dict(color='lightgrey', size=8, opacity=0.5, line=dict(width=0.5, color='DarkGrey')),
            name='No AUC Data',
            text=df_missing['Treatment'],
            customdata=np.stack((df_missing['Plate'], df_missing['Well_ID']), axis=-1),
            hovertemplate="<b>%{text}</b><br>Plate: %{customdata[0]}<br>Well: %{customdata[1]}<br>AUC: N/A<extra></extra>"
        )
    )
    
    fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=0.5, color='DarkGrey')), selector=dict(mode='markers'))
    
    fig.update_layout(
        width=950, height=850,
        coloraxis_colorbar=dict(title="AUC Score"),
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    save_name = f"UMAP_AUC_{config['name'].replace(' ', '_')}.html"
    save_path = os.path.join(OUTPUT_DIR, save_name)
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    fig.show()

In [ ]:
#annotation with names

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt
from matplotlib.path import Path

# --- 0. ADAPTABLE STYLE PARAMETERS ---
# Axis Style
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

# Legend Style
LEGEND_FONT_SIZE = 14     
LEGEND_TITLE_SIZE = 16    

# Plot Marker Style
MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      
DOT_LABEL_SIZE = 18       # <--- NEW: Size of the names next to the dots
PNG_RESOLUTION_SCALE = 7  

# --- 1. SETUP & THE COLOR DICTIONARY ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts_T0_T1.csv")
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "T0T1_annotated_with_Names")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                         
    "biosynthesis of fatty acids": "#9467bd",     
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "biosynthesis of isoprenoids": "#7f7f7f",     
    "cell division": "#bcbd22",                   
    "ribosomal proteins": "#17becf",              
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                       
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#808080",  
    "Baseline (T0)": "#808080",  
    "Unknown/Other": "#D3D3D3"   
}

# --- 2. DATA LOADING & ANNOTATION ---
df_raw = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df_raw.columns = [str(c) for c in df_raw.columns]

df = df_raw.copy()
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

df['SubtiWiki Annotation 4'] = df['SubtiWiki Annotation 4'].astype(str).str.strip()
df['SubtiWiki Annotation 3'] = df['SubtiWiki Annotation 3'].astype(str).str.strip()

df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].replace('nan', np.nan).fillna(df['SubtiWiki Annotation 3'].replace('nan', np.nan)).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

is_ctrl = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[(df['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
df.loc[is_ctrl, 'Display_Category'] = 'Control Group'

# Legend Order Logic
unique_in_data = df['Display_Category'].unique()
other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["Unknown/Other"]

auto_pal = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
color_lookup = MANUAL_COLORS.copy()
for i, cat in enumerate(other_cats):
    color_lookup[cat] = auto_pal[i % len(auto_pal)]

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df_raw.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Processing Logic Percentages + Names for: {config['name']}...")
    
    # UMAP
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Save Coords
    meta_cols = ['Plate', 'Well_ID', 'Treatment', 'Timepoint', 'Display_Category', 'Cell_Count', 'UMAP1', 'UMAP2']
    df_plot[[c for c in meta_cols if c in df_plot.columns]].to_csv(os.path.join(COORD_DIR, f"Coords_Logic_{config['name'].replace(' ', '_')}.csv"), index=False)

    fig = go.Figure()

    # --- 5. KDE Logic Percentages ---
    ctrl_df = df_plot[df_plot['Display_Category'] == 'Control Group']
    if not ctrl_df.empty:
        x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
        points_array = np.vstack([x_pts, y_pts]).T
        kde = st.gaussian_kde(points_array.T)
        point_densities = kde(points_array.T)
        
        target_inclusions = [95, 70, 45, 20]
        percentile_thresholds = [100 - t for t in target_inclusions]
        levels = [np.percentile(point_densities, p) for p in percentile_thresholds]
        
        pad = 2
        x_grid = np.linspace(x_pts.min() - pad, x_pts.max() + pad, 150)
        y_grid = np.linspace(y_pts.min() - pad, y_pts.max() + pad, 150)
        X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
        Z_mesh = kde(np.vstack([X_mesh.ravel(), Y_mesh.ravel()])).reshape(X_mesh.shape)

        fig_temp, ax_temp = plt.subplots()
        cs = ax_temp.contour(X_mesh, Y_mesh, Z_mesh, levels=levels)
        
        for i, (level_val, target_pct) in enumerate(zip(levels, target_inclusions)):
            segments = cs.allsegs[i]
            for j, seg in enumerate(segments):
                fig.add_trace(go.Scatter(
                    x=seg[:, 0], y=seg[:, 1], mode='lines',
                    line=dict(color='#808080', width=1.2),
                    name=f'Control Group ({target_pct}%)' if i == 0 and j == 0 else 'Control_Inner',
                    legendgroup='Control Group', showlegend=False, hoverinfo='skip'
                ))
        plt.close(fig_temp)

    # --- 6. Plot Categories with Text Names ---
    remaining_cats = [c for c in final_legend_order[::-1] if c != "Control Group"]

    for cat in remaining_cats:
        if cat not in df_plot['Display_Category'].values: continue
        sub_df = df_plot[df_plot['Display_Category'] == cat]
        clr = color_lookup.get(cat, "#000000")

        # Generate Detailed Hover Label
        hover_labels = []
        for _, r in sub_df.iterrows():
            txt = (f"<b>Category:</b> {cat}<br>"
                   f"<b>Treatment:</b> {r.get('Treatment','N/A')}<br>"
                   f"<b>Timepoint:</b> {r.get('Timepoint','N/A')}<br>"
                   f"<b>Plate:</b> {r.get('Plate','N/A')}<br>"
                   f"<b>Well:</b> {r.get('Well_ID','N/A')}<br>"
                   f"<b>Cell Count:</b> {r.get('Cell_Count','N/A')}")
            hover_labels.append(txt)

        # ADD TRACE WITH TEXT MODE
        fig.add_trace(go.Scatter(
            x=sub_df['UMAP1'], 
            y=sub_df['UMAP2'], 
            mode='markers+text',            # <--- ADDED +text
            text=sub_df['Treatment'],        # <--- Name shown next to dot
            textposition='top center',       # <--- Position of the name
            textfont=dict(size=DOT_LABEL_SIZE, color='black'),
            name=cat, 
            legendgroup=cat, 
            showlegend=False,
            marker=dict(size=MARKER_SIZE, color=clr, symbol='circle',
                        opacity=MARKER_OPACITY, line=dict(width=0.4, color='white')),
            customdata=hover_labels,         # Use customdata for rich hover
            hovertemplate="%{customdata}<extra></extra>"
        ))

    # --- 7. Legend Order Override ---
    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            name = trace.name if trace.name else ""
            if name.startswith('Control Group') and 'Control Group' not in seen:
                trace.showlegend = True
                trace.name = 'Control Group'
                seen.add('Control Group')
                trace.legendrank = final_legend_order.index('Control Group')
            elif name == cat and cat not in seen:
                trace.showlegend = True
                seen.add(cat)
                trace.legendrank = final_legend_order.index(cat)

    # --- 8. Layout ---
    fig.update_layout(
        template='plotly_white', width=1600, height=1000, # Widened to fit labels
        margin=dict(l=80, r=20, b=80, t=80), font=dict(family="Arial"),
        legend=dict(title_text='<b>Pathways</b>', title_font=dict(size=LEGEND_TITLE_SIZE),
                    x=1.02, y=1, font=dict(size=LEGEND_FONT_SIZE), itemsizing='constant'),
        xaxis=dict(title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                   showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, showgrid=False,
                   ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
                   tickfont=dict(size=AXIS_TICK_FONT_SIZE), scaleanchor="y", scaleratio=1),
        yaxis=dict(title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                   showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, showgrid=False,
                   ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
                   tickfont=dict(size=AXIS_TICK_FONT_SIZE))
    )

    base_name = f"UMAP_Logic_Names_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=PNG_RESOLUTION_SCALE)

print("\nDone! Plots with KDE Logic Percentages and Treatment Names are saved.")

In [ ]:
#mask overlay: 3 april still to be adapted

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# ==========================================
# 1. SETUP & DATA MAPPING
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "2april", "vettedcellcounts_2april_T0_T1.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings for consistency
df.columns = [str(c) for c in df.columns]

# Map Area Data
area_path = r'D:\Thesis\final\area\master_median_areas_per_site.csv'
area_df = pd.read_csv(area_path)
area_df['Well_ID'] = area_df['Plate'] + "_" + area_df['Well']
area_map = area_df.groupby('Well_ID')['Median_Area'].median().to_dict()

# Add Area to main dataframe and handle missing values
df['Median_Area_Size'] = df['Well_ID'].map(area_map)
df['Median_Area_Size'] = df['Median_Area_Size'].fillna(df['Median_Area_Size'].mean())

# Identify 'Type' for symbols
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2april", "UMAP_2april_maskarea")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ==========================================
# 2. FEATURE SELECTION
# ==========================================
vetted_features = [f for f in df.columns if f.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 3. RUN UMAP & PLOT WITH ROBUST COLOR SCALE
# ==========================================
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # --- ROBUST COLOR SCALING ---
    # We find the 95th percentile to prevent outliers from squashing the gradient
    color_min = df_plot['Median_Area_Size'].min()
    color_max = df_plot['Median_Area_Size'].quantile(0.99) 
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Median_Area_Size',     
        symbol='Type',                
        hover_name='Treatment',
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True,
            'Median_Area_Size': ':.2f',
            'UMAP1': False, 
            'UMAP2': False
        },
        range_color=[color_min, color_max], # This forces the gradient to ignore outliers
        color_continuous_scale='Viridis',
        title=f"UMAP: {config['name']} (Color Cap at 95th Percentile)",
        template='plotly_white'
    )
    
    # Adjust point appearance
    fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=0.5, color='DarkGrey')))
    
    # Clean up layout
    fig.update_layout(
        width=950, 
        height=850,
        coloraxis_colorbar=dict(
            title="Median Area",
            ticksuffix="+" if color_max < df_plot['Median_Area_Size'].max() else ""
        ),
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save and Show
    save_name = f"UMAP_FIXED_GRADIENTcap_{config['name'].replace(' ', '_')}.html"
    save_path = os.path.join(OUTPUT_DIR, save_name)
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    fig.show()

In [ ]:
#AUC overlay, other growth characteristics overlay

In [ ]:
#antibiotics#
#######






#

In [ ]:
#aggregation antibiotics 

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
PLATES = ["PLATE6_T1","PLATE7_T1"]

# Paths for the three separate outputs
OUTPUT_CSV_MEDIAN = os.path.join(PROJECT_ROOT,"31march", "antibiotics_aggregated_wells_median.csv")
OUTPUT_CSV_MEAN   = os.path.join(PROJECT_ROOT,"31march", "antibiotics_aggregated_wells_mean.csv")
OUTPUT_CSV_STD    = os.path.join(PROJECT_ROOT,"31march", "antibiotics_aggregated_wells_std.csv")

CELL_COUNT_THRESHOLD = 0  
TREATMENT_COL = "Treatment"

all_plates_median = []
all_plates_mean = []
all_plates_std = []
all_cell_counts = [] 

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT, "features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT, "metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    # --- AGGREGATION & THRESHOLDING STEP ---
    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_cell_count = all_cells_in_well.shape[0]
        all_cell_counts.append(well_cell_count)

        if well_cell_count >= CELL_COUNT_THRESHOLD:
            # Calculate aggregations
            well_median = np.median(all_cells_in_well, axis=0)
            well_mean   = np.mean(all_cells_in_well, axis=0)
            well_std    = np.std(all_cells_in_well, axis=0)
            
            base_info = {
                "Plate": plate_id, 
                "Well_ID": well_id, 
                "Treatment": well_to_treatment[well_id],
                "Cell_Count": well_cell_count
            }
            
            # Efficiently map feature indices to values
            feat_cols = {idx: val for idx, val in enumerate(well_median)}
            all_plates_median.append({**base_info, **feat_cols})
            
            feat_cols_mean = {idx: val for idx, val in enumerate(well_mean)}
            all_plates_mean.append({**base_info, **feat_cols_mean})
            
            feat_cols_std = {idx: val for idx, val in enumerate(well_std)}
            all_plates_std.append({**base_info, **feat_cols_std})

# Convert to DataFrames
df_median = pd.DataFrame(all_plates_median)
df_mean   = pd.DataFrame(all_plates_mean)
df_std    = pd.DataFrame(all_plates_std)

# Helper function to reorder columns consistently
def reorder_cols(df):
    if df.empty: return df
    meta_cols = ["Plate", "Well_ID", "Treatment", "Cell_Count"]
    feat_cols = sorted([c for c in df.columns if c not in meta_cols])
    return df[meta_cols + feat_cols]

df_median = reorder_cols(df_median)
df_mean   = reorder_cols(df_mean)
df_std    = reorder_cols(df_std)

print(f"\nAggregation complete.")

# ==========================================
# 2. SAVE TO CSVs
# ==========================================
df_median.to_csv(OUTPUT_CSV_MEDIAN, index=False)
df_mean.to_csv(OUTPUT_CSV_MEAN, index=False)
df_std.to_csv(OUTPUT_CSV_STD, index=False)

print(f"Median data saved: {OUTPUT_CSV_MEDIAN}")
print(f"Mean data saved:   {OUTPUT_CSV_MEAN}")
print(f"Std Dev data saved: {OUTPUT_CSV_STD}")

In [ ]:
#min5 wells antiiobitcs (maar is niet nodig)

In [ ]:
import pandas as pd
import os

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_DIR = os.path.join(PROJECT_ROOT, "31march")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "31march_filtered")

# Create the directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. DEFINE YOUR NEW THRESHOLD
STRICT_THRESHOLD = 5 

# List of the aggregation files you created in the previous step
FILES_TO_FILTER = [
    "antibiotics_aggregated_wells_median.csv",
    "antibiotics_aggregated_wells_mean.csv",
    "antibiotics_aggregated_wells_std.csv"
]

# 3. PROCESSING LOOP
for file_name in FILES_TO_FILTER:
    file_path = os.path.join(INPUT_DIR, file_name)
    
    if not os.path.exists(file_path):
        print(f"Skipping {file_name}: File not found.")
        continue

    print(f"\n--- Processing {file_name} ---")
    df = pd.read_csv(file_path)

    # Identify the wells to keep and remove
    # We use 'Cell_Count' which we added during the aggregation step
    filtered_df = df[df['Cell_Count'] >= STRICT_THRESHOLD].copy()
    removed_df = df[df['Cell_Count'] < STRICT_THRESHOLD].copy()

    # 4. REPORT
    print(f"Original wells: {len(df)}")
    print(f"Wells kept:     {len(filtered_df)}")
    print(f"Wells removed:  {len(removed_df)}")

    if not removed_df.empty and "median" in file_name:
        # Just show the list once (for the median file) to avoid clutter
        print(f"\nExample of removed wells (Count < {STRICT_THRESHOLD}):")
        print(removed_df[['Plate', 'Well_ID', 'Treatment', 'Cell_Count']].head(10).to_string(index=False))

    # 5. SAVE DATA
    # Renaming the output to include the 'min5' suffix
    output_name = file_name.replace(".csv", "_min5.csv")
    output_path = os.path.join(OUTPUT_DIR, output_name)
    filtered_df.to_csv(output_path, index=False)
    
    print(f"Filtered data saved to: {output_path}")

print("\nAll filtering tasks complete!")

In [ ]:
#zelf die paar slechte verwijderd in de juist ifle

In [ ]:
#puur mean plotten met antibiotica anotatie

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "31march_filtered", "antibiotics_aggregated_wells_mean_juist.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2aprilAntibiotics", "UMAP_all_mean")
os.makedirs(OUTPUT_DIR, exist_ok=True)

df.columns = [str(c) for c in df.columns]

# 2. PRE-PROCESSING
# Extract Base Treatment (e.g., 'vancomycin')
df['Treatment_Base'] = df['Treatment'].astype(str).str.split('_').str[0]

# Define Symbol and special Color logic
def assign_plot_logic(row):
    treatment = str(row['Treatment']).lower()
    is_control = any(ctrl in treatment for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        # Controls get a unique label so we can color them black
        color_group = "Control (nosgrna)"
        symbol = "circle" if row['Plate'] == "PLATE6_T1" else "x"
    else:
        # Mutants use their antibiotic name for color
        color_group = row['Treatment_Base']
        symbol = "circle" if row['Plate'] == "PLATE6_T1" else "x"
        
    return pd.Series([color_group, symbol])

df[['Color_Group', 'Symbol_Type']] = df.apply(assign_plot_logic, axis=1)

# Create a color map to force Controls to be Black
unique_treatments = df['Color_Group'].unique()
color_map = {t: px.colors.qualitative.Alphabet[i % 26] for i, t in enumerate(unique_treatments)}
color_map["Control (nosgrna)"] = "#000000"  # Hex for pure black

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 4. RUN UMAP
for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df_plot, x='UMAP1', y='UMAP2', 
        color='Color_Group',
        symbol='Symbol_Type',
        color_discrete_map=color_map, # Forces control to black
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'Cell_Count': True},
        title=f"UMAP: {config['name']} (Black Controls: Dot=P6, X=P7)",
        template='plotly_white'
    )
    
    # Final styling
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    # Make the Black Controls slightly larger and fully opaque to stand out
    fig.update_traces(marker=dict(size=10, opacity=1.0), selector=dict(marker_color='#000000'))
    
    fig.update_layout(width=1000, height=800, legend_title_text='Treatments & Controls')
    
    save_path = os.path.join(OUTPUT_DIR, f"UMAP_BlackControls2_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    fig.show()

In [ ]:
# antibiotica classe anotatie

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "31march_filtered", "antibiotics_aggregated_wells_mean_juist.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2aprilAntibiotics", "UMAP_MOA_Groups")
os.makedirs(OUTPUT_DIR, exist_ok=True)

df.columns = [str(c) for c in df.columns]

# --- NEW: Define MOA Mapping ---
moa_map = {
    'vancomycin': 'Cell Wall', 
    'cefalexin': 'Cell Wall', 
    'cefotaxime': 'Cell Wall',
    'rifampicin': 'RNA',
    'ciprofloxacin': 'DNA',
    'gentamycin': 'Ribosome', 
    'kanamycin': 'Ribosome', 
    'tetracycline': 'Ribosome',
    'Control (nosgrna)': 'Control'
}

# 2. PRE-PROCESSING
df['Treatment_Base'] = df['Treatment'].astype(str).str.split('_').str[0]

def assign_plot_logic(row):
    treatment = str(row['Treatment']).lower()
    is_control = any(ctrl in treatment for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        color_group = "Control (nosgrna)"
    else:
        color_group = row['Treatment_Base']
    
    # Assign the MOA based on the treatment base
    moa = moa_map.get(color_group, 'Unknown')
    symbol = "circle" if row['Plate'] == "PLATE6_T1" else "x"
    return pd.Series([color_group, symbol, moa])

df[['Color_Group', 'Symbol_Type', 'MOA']] = df.apply(assign_plot_logic, axis=1)

# Create a color map for the MOA groups
unique_moas = df['MOA'].unique()
# Using a clear qualitative palette
color_map = {m: px.colors.qualitative.Bold[i % 10] for i, m in enumerate(unique_moas)}
color_map["Control"] = "#000000"  # Force Control MOA to Black

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 4. RUN UMAP
for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    
    # Keeping your original settings
    embedding = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df_plot, x='UMAP1', y='UMAP2', 
        color='MOA',  # Color by the biological target
        symbol='Symbol_Type',
        color_discrete_map=color_map,
        hover_name='Treatment', # Allows you to see specific antibiotic on hover
        hover_data={'Plate': True, 'Well_ID': True, 'Color_Group': True},
        title=f"UMAP: {config['name']} grouped by MOA (Black=Control)",
        template='plotly_white'
    )
    
    # Final styling
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    # Make the Black Controls stand out
    fig.update_traces(marker=dict(size=10, opacity=1.0), selector=dict(marker_color='#000000'))
    
    fig.update_layout(width=1000, height=800, legend_title_text='Mechanism of Action')
    
    save_path = os.path.join(OUTPUT_DIR, f"UMAP_MOA_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    fig.show()

In [ ]:
#antibiotica op knockdown mappen

In [ ]:
#feature selection met antibioticaplaten erbij

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "2april", "aggregated_wells_median_min5_met67.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2april")
CONTROL_LABEL = "no_sgRNA" 

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Loading raw data...")
df_raw_full = pd.read_csv(INPUT_CSV)
df_raw_full.columns = [str(c) for c in df_raw_full.columns]

# --- STEP: FILTER AND VIRTUAL MAPPING ---
print("Filtering out T2 samples and remapping PLATE6/7_T1 to T0...")

# 1. Keep only T0 and T1
df_raw = df_raw_full[df_raw_full['Plate'].str.contains('T0|T1')].copy()

# 2. Create 'Virtual_TP' for calculation logic
# Default: take last 2 characters (T0 or T1)
df_raw['Virtual_TP'] = df_raw['Plate'].str[-2:]

# 3. Explicitly remap PLATE6_T1 and PLATE7_T1 to T0 for feature selection
mask = df_raw['Plate'].isin(['PLATE6_T1', 'PLATE7_T1'])
df_raw.loc[mask, 'Virtual_TP'] = 'T0'

# Separate Metadata and Features
# We include Virtual_TP in metadata so it isn't treated as a feature
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count', 'Virtual_TP']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    
    # Calculate plate medians while keeping the Virtual_TP mapping
    plate_medians = ctrls.groupby(['Plate', 'Virtual_TP'])[features].median().reset_index()

    tp_batch_noises = []
    # Loop through the virtual groups (T0 and T1)
    for tp in plate_medians['Virtual_TP'].unique():
        tp_subset = plate_medians[plate_medians['Virtual_TP'] == tp][features]
        
        if len(tp_subset) > 1:
            noise = tp_subset.std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        print("Warning: Not enough plates per virtual timepoint. Returning input features.")
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# --- STEP 0: GLOBAL VARIANCE FILTER ---
print(f"\nRunning Step 0: Global Variance Filter (std > 0.01)...")
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
print(f"Removed {len(feature_cols) - len(active_features)} low-variance features.")

# --- STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"Running Step 1: Within-Plate Consistency (Filtering to top 2000)...")
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=2000)
df_step1 = df_raw[metadata_cols + step1_features]

# --- STEP 2: ACROSS-PLATE STABILITY ---
print(f"Running Step 2: Across-Plate Stability (Filtering to top 200)...")
# This now uses the Virtual_TP logic internally
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
df_step2 = df_step1[metadata_cols + step2_features]

# --- STEP 3: REDUNDANCY REMOVAL ---
print(f"Running Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
# We drop Virtual_TP for the final output to keep the format clean
df_final = df_step2[metadata_cols + final_feature_list].drop(columns=['Virtual_TP'])
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts_2april_T0_T1_met67.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE (T0 & T1 ONLY)")
print(f"Note: PLATE6_T1 & PLATE7_T1 treated as T0 for stability calculation.")
print(f"Original rows (including T2): {len(df_raw_full)}")
print(f"Filtered rows (T0 & T1): {len(df_raw)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved results to: {output_path}")
print("="*40)

In [ ]:
#antibiotica plotten op de knockdown

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "2april", "vettedcellcounts_2april_T0_T1_met67color")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "2april", "vettedcellcounts_2april_T0_T1_met67.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

print("Loading data...")
df_raw = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df_raw.columns = [str(c) for c in df_raw.columns]

# --- 2. GLOBAL VARIANCE FILTER ---
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c.isdigit()]
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
df = df_raw[metadata_cols + active_features].copy()

# --- 3. MERGE & CATEGORIZATION ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3']).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Identify groups
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
is_special_plate = df['Plate'].str.contains('PLATE6|PLATE7', case=False, na=False)

# NEW: Create a simplified Treatment name (everything before the first underscore)
# e.g., "vancomycin_50" becomes "vancomycin"
df['Treatment_Base'] = df['Treatment'].str.split('_').str[0]

# CATEGORIZATION LOGIC
df['Display_Category'] = df['Effective_Annotation']

# If it's PLATE6/7 and NOT a control, use the BASE Treatment name for coloring
df.loc[is_special_plate & ~is_control, 'Display_Category'] = df['Treatment_Base']

# Standard overrides
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control) & (~is_special_plate), 'Display_Category'] = 'Baseline (T0)'

# --- 4. GOLDEN ANGLE COLOR MAPPING ---
exclude = ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in exclude])
num_cats = len(all_cats)

if num_cats > 0:
    base_palette = px.colors.sample_colorscale("Turbo", [i/255 for i in range(256)])
    golden_ratio_conjugate = 0.618033988749895
    h_values = [(i * golden_ratio_conjugate) % 1 for i in range(num_cats)]
    max_contrast_palette = [base_palette[int(h * 255)] for h in h_values]
    color_map = {cat: max_contrast_palette[i] for i, cat in enumerate(all_cats)}
else:
    color_map = {}

color_map['no_sgrna'] = '#EBEBEB'
color_map['Baseline (T0)'] = '#B0B0B0'
color_map['Unknown/Other'] = '#222222'

# --- 5. EXECUTION ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Define shapes
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'
    df.loc[is_special_plate, 'Point_Shape'] = 'diamond'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        symbol_map={"circle": "circle", "x": "x", "square": "square", "diamond": "diamond"},
        hover_name='Treatment', # Keep full name in hover
        hover_data=['Plate', 'Effective_Annotation'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    # Trace/Legend clean up
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Styling
    fig.update_traces(marker=dict(opacity=0.9, line=dict(width=0.5, color='white'))) 
    
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='x'))
    fig.update_traces(marker=dict(size=10, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square'))
    
    # Apply Large Diamond styling (colored by Treatment_Base)
    fig.update_traces(
        marker=dict(size=16, line=dict(width=1.5, color='black')), 
        selector=dict(marker_symbol='diamond')
    )
    
    fig.update_layout(
        width=1400, height=900,
        legend_title_text='Group / Compound (Diamonds)',
        xaxis=dict(title="UMAP 1", showline=True, linewidth=2, linecolor='black', showgrid=False),
        yaxis=dict(title="UMAP 2", showline=True, linewidth=2, linecolor='black', showgrid=False)
    )
    
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))

print(f"Done. Plate 6/7 points are diamonds colored by compound name.")